# Arbeitspaket (AP) 1: Management & Nutzung Relationale Daten
# Kernel 3.12.11

## Python Libraries und Settings

In [1]:
# Libraries
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings("ignore")

# Working directory
print(os.getcwd())

/workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01/Zusatzmaterial


## <b>Vorbereitung (Dieser Teil wird <u>nicht</u> bewertet!)</b>

#### <b>1.) Starten Sie eine GitHub Codespaces Instanz auf Basis Ihres Forks des folgenden GitHub Repositories:</b>

##### GitHub-Repository: https://github.com/mario-gellrich-zhaw/python_postgresql_pgadmin


#### <b>2.) Erstellen und Testen Sie eine Datenbankverbindung mit der 'postgres' Datenbank.</b>

In [2]:
#Du sollst den Codespace des python_postgresql_pgadmin-Repos starten, weil er die benötigte PostgreSQL-/Python-Umgebung bereitstellt
#Die Bike-Shop-Dateien sind zwar separat, sollen aber offenbar innerhalb genau dieses Codespaces ausgeführt werden
#D.h. bike shop data ist lokal, aber wir benötigen die Ressourcen des Online Codespaces

# Set up für Datenbankverbindung
user = "pgadmin"
password = "geheim"
host = "localhost"
port = "5432"
database = "postgres"

# Erstellen der Connection URL
db_connection_url = "postgresql://" + user + ":" + password +\
                    "@" + host + ":" + port + "/" + database

# Erstellen SQLAlchemy Engine
engine = create_engine(db_connection_url)

# Test der Datenbankverbindung
with engine.connect() as connection:
    result = connection.execute(text('SELECT current_database()'))
    print(result.fetchone())

# Verbindung trennen
engine.dispose()

('postgres',)


#### <b>3.) Lesen Sie die Tabellenblätter aus 'bike_shop_data.xlsx' ein, und speichern Sie diese in separaten DataFrames.</b>

In [3]:
# Pfad zur .xlsx Datei
path = 'bike_shop_data.xlsx'

# Einlesen der Daten aus Excel File
df_bicycles = pd.read_excel(path, sheet_name='Bicycles')
df_customers = pd.read_excel(path, sheet_name='Customers')
df_sales = pd.read_excel(path, sheet_name='Sales')
df_salespersons = pd.read_excel(path, sheet_name='SalesPersons')
df_mechanics = pd.read_excel(path, sheet_name='Mechanics')
df_servicerecords = pd.read_excel(path, sheet_name='ServiceRecords')

# Anpassen der Spaltennamen zu 'lowercase'
df_bicycles.columns = df_bicycles.columns.str.lower()
df_customers.columns = df_customers.columns.str.lower()
df_sales.columns = df_sales.columns.str.lower()
df_salespersons.columns = df_salespersons.columns.str.lower()
df_mechanics.columns = df_mechanics.columns.str.lower()
df_servicerecords.columns = df_servicerecords.columns.str.lower()


#### <b>4.) Schreiben Sie die erstellten DataFrames in separate Tabellen in Ihrer Datenbank.</b>

In [4]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# Schreiben der DataFrames in die Datenbank (ohne zusätzliche Index-Spalte)
df_bicycles.to_sql('bicycles', engine, if_exists='replace', index=False)
df_customers.to_sql('customers', engine, if_exists='replace', index=False)
df_sales.to_sql('sales', engine, if_exists='replace', index=False)
df_salespersons.to_sql('salespersons', engine, if_exists='replace', index=False)
df_mechanics.to_sql('mechanics', engine, if_exists='replace', index=False)
df_servicerecords.to_sql('servicerecords', engine, if_exists='replace', index=False)

# Datenbankverbindung trennen
engine.dispose()

<div>

#### <b>Hinweis!</b>

#### <b>Falls etwas mit der PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für die folgenden Aufgaben die SQLite-Datenbank 'bike_shop.db'.</b>

#### <b>Erstellen Sie dazu die Engine in den Aufgaben-Zellen mit `create_engine('sqlite:///bike_shop.db')` statt mit `create_engine(db_connection_url)`.</b>

Das folgende Python-Codebeispiel demonstriert den Verbindungsaufbau mit der SQLite-Datenbank:

```python
# Engine für Datenbankverbindung erstellen
engine = create_engine('sqlite:///bike_shop.db')

# SQL Abfrage
df_sub = pd.read_sql_query("""SELECT *
                              FROM bicycles;""",
                          con=engine)

# Schliessen der Datenbankverbindung
engine.dispose()

df_sub
```

<b>Unterschiede SQLite gegenüber PostgreSQL (nur relevant, falls Sie SQLite verwenden):</b>
<ul>
  <li>Jahr aus einem Datum: <code>CAST(strftime('%Y', ServiceDate) AS INTEGER)</code> statt <code>EXTRACT(YEAR FROM ServiceDate)</code>.</li>
  <li>Datumswerte sind in SQLite als Text mit Uhrzeit gespeichert (z.B. '2023-12-31 00:00:00.000000'). Verwenden Sie für Datumsbereiche <code>DATE(SaleDate) BETWEEN '2023-03-01' AND '2023-12-31'</code>, sonst fehlt der letzte Tag.</li>
  <li>Runden: <code>ROUND(AVG(Price), 2)</code> funktioniert direkt, ein <code>CAST(... AS NUMERIC)</code> ist nicht nötig (aber auch nicht falsch).</li>
</ul>
</div>

## <b>Aufgaben (Dieser Teil wird bewertet!)</b>
<b>Hinweise zu den folgenden Aufgabenstellungen:</b>
<ul>
  <li>Erstellen Sie je Aufgabe eine SQL-Abfrage und integrieren Sie diese in Ihren Python Code.</li>
  <li>Speichern Sie je Aufgabe die Ergebnistabelle in einem DataFrame und stellen Sie diesen ganz oder in Teilen (z.B. erste 5 Zeilen) dar.</li>
  <li>Die Gross-/Kleinschreibung der Spaltennamen in der Ergebnistabelle wird nicht bewertet. PostgreSQL wandelt Spaltennamen und Aliase ohne Anführungszeichen automatisch in Kleinbuchstaben um (z.B. <code>BicycleCount</code> → <code>bicyclecount</code>).</li>
  <li>Runden in PostgreSQL: <code>ROUND()</code> mit Nachkommastellen funktioniert nur mit dem Datentyp NUMERIC, z.B. <code>ROUND(CAST(AVG(Price) AS NUMERIC), 2)</code>.</li>
  <li>Prozentzeichen in SQL-Abfragen (z.B. bei <code>LIKE</code>) müssen im Python-Code verdoppelt werden: <code>LIKE 'Gear%%'</code> statt <code>LIKE 'Gear%'</code>, sonst bricht die Abfrage mit einem Fehler ab.</li>
</ul>
<b style="color:red;">Beachten Sie, dass für die Punktevergabe auch die weiteren Anforderungen zu den Fragen unter 'Details zur Aufgabenstellung' herangezogen werden.</b>

#### <b>Aufgabe (1): Bestimmen Sie die Anzahl Bicycles pro Status.</b>
<b>Details zur Aufgabenstellung:</b>  
- Benötigte Tabelle: Bicycles  
- Benötigte Spalten: Status  
- Die Ergebnistabelle soll die Spalten enthalten: Status, BicycleCount.  
- Sortieren Sie die Ausgabe nach BicycleCount (absteigend).  

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [5]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_sub = pd.read_sql_query('''
    SELECT Status,
    COUNT (*) AS BicycleCount
    FROM Bicycles
    GROUP BY Status
    ORDER BY BicycleCount DESC;
    ''', con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_sub

,status,bicyclecount
0,Sold,684
1,Available,672
2,In Maintenance,644


#### <b>Aufgabe (2): Listen Sie den durchschnittlichen Listenpreis (Spalte Price) der Bicycles für jede Kombination aus Marke und Modell auf.</b>
<b>Details zur Aufgabenstellung:</b>
- Benötigte Tabelle: Bicycles.
- Benötigte Spalten: Price, Make, Model.
- Die Ergebnistabelle soll die Spalten Make, Model und AveragePrice enthalten.
- Runden Sie den Durchschnittspreis auf zwei Nachkommastellen.
- Sortieren Sie die Ausgabe absteigend nach Make und absteigend nach AveragePrice.

<b>Tipp:</b>
- Eine Sortierung über zwei Spalten kann z.B. mittels 'ORDER BY Spalte A DESC, Spalte B DESC' erfolgen.

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [6]:
display(df_bicycles)



,bicycleid,make,model,year,framenr,price,status
0,C7ZGWFLBUV,Trek,Domane,2019,A15 0LX,5066.94,In Maintenance
1,C875HZ1AG6,Merida,Reacto,2021,435N,5529.38,Sold
2,CDONEWSWFP,Merida,Big Nine,2021,QBA-572,1518.33,In Maintenance
3,C2UQZP803C,Trek,Marlin,2023,5-92778W,5038.99,In Maintenance
4,C2WFIK6WEP,Merida,Scultura,2020,300 VLT,1548.19,Sold
...,...,...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2020,911 LYI,2376.02,Available
1996,CP3F48BE9Y,Merida,Scultura,2022,YPU 135,2833.36,Available
1997,CHBXQ3SSNU,Merida,Scultura,2019,059-GNM,1629.91,In Maintenance
1998,CP5CTME2ZD,Trek,Marlin,2021,02V 6510,6176.55,In Maintenance


In [7]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_subzwei = pd.read_sql_query('''
SELECT Make, Model,
ROUND(CAST(AVG(Price) AS NUMERIC), 2) AS AveragePrice
FROM Bicycles
GROUP BY Make, Model
ORDER BY Make DESC, AveragePrice DESC
''', con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_subzwei

,make,model,averageprice
0,Trek,Marlin,4666.03
1,Trek,Domane,4466.53
2,Trek,Emonda,4119.89
3,Specialized,Rockhopper,4742.53
4,Specialized,Roubaix,4504.83
5,Specialized,Allez,4128.30
6,Scott,Aspect,4297.69
7,Scott,Addict,4178.61
8,Scott,Spark,4091.72
9,Santa Cruz,Bronson,4747.68


#### <b>Aufgabe (3): Ermitteln Sie, wie viele Bicycles pro Marke einen Preis von 2'000.- bis 4'000.- USD haben.</b>
<b>Details zur Aufgabenstellung:</b>
- Benötigte Tabelle: Bicycles.
- Benötigte Spalten: Make, Price.
- Die Ergebnistabelle soll die Spalten Make und NumberOfBikes enthalten.
- Sortieren Sie die Ausgabe absteigend nach NumberOfBikes.

<b>Tipp:</b>
- Ein Werte- oder Datumsbereich kann mittels 'BETWEEN' und 'AND' definiert werden.
- Dabei ist 'BETWEEN' inklusive der verwendeten Angaben an beiden Enden.
- z.B. WHERE Value BETWEEN 22 AND 45 schliesst 22 und 45 mit ein.
- z.B. WHERE Datum BETWEEN '2021-01-01' AND '2021-12-31' beinhaltet in PostgreSQL das gesamte Jahr 2021 (sofern die Datumswerte keine Uhrzeit enthalten, was in diesen Daten der Fall ist).

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [8]:
display(df_bicycles)

,bicycleid,make,model,year,framenr,price,status
0,C7ZGWFLBUV,Trek,Domane,2019,A15 0LX,5066.94,In Maintenance
1,C875HZ1AG6,Merida,Reacto,2021,435N,5529.38,Sold
2,CDONEWSWFP,Merida,Big Nine,2021,QBA-572,1518.33,In Maintenance
3,C2UQZP803C,Trek,Marlin,2023,5-92778W,5038.99,In Maintenance
4,C2WFIK6WEP,Merida,Scultura,2020,300 VLT,1548.19,Sold
...,...,...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2020,911 LYI,2376.02,Available
1996,CP3F48BE9Y,Merida,Scultura,2022,YPU 135,2833.36,Available
1997,CHBXQ3SSNU,Merida,Scultura,2019,059-GNM,1629.91,In Maintenance
1998,CP5CTME2ZD,Trek,Marlin,2021,02V 6510,6176.55,In Maintenance


In [9]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_subdrei = pd.read_sql_query('''
SELECT make,
COUNT (*) AS NumberOfBikes
FROM Bicycles
WHERE price BETWEEN 2000 AND 4000
GROUP BY make
ORDER BY NumberOfBikes DESC

''', con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_subdrei

,make,numberofbikes
0,Cannondale,66
1,Giant,64
2,Specialized,57
3,Bianchi,57
4,Santa Cruz,54
5,Cervelo,54
6,Cube,52
7,Merida,50
8,Scott,45
9,Trek,42


#### <b>Aufgabe (4): Ermitteln Sie die durchschnittlichen Kosten für den Service 'Gear shifting adjustment' pro Jahr.</b>
<b>Details zur Aufgabenstellung:</b>
- Benötigte Tabelle: ServiceRecords
- Benötigte Spalten: ServiceDate, ServiceDescription, Cost
- Die Ergebnistabelle soll die Spalten Year und AverageServiceCosts enthalten.
- Runden Sie AverageServiceCosts auf zwei Nachkommastellen.
- Sortieren Sie die Ausgabe absteigend nach AverageServiceCosts.

<b>Tipp:</b>
- Mit CAST(EXTRACT(YEAR FROM ServiceDate) AS INTEGER) AS Year können Sie das Jahr als Ganzzahl aus ServiceDate extrahieren.

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [10]:
df_servicerecords

,serviceid,bicycleid,servicedate,servicedescription,cost,mechanicid
0,SRPK835UWM,CRK1AC8K7M,2020-02-21,General inspection & safety check,58.60,MTA6ED59YC
1,SR4CZM07VB,C88VOY5YSJ,2019-10-27,"Tire replacement (per tire, excl. tire)",17.24,MCNI3BOSV1
2,SRKMA0NOB6,CVS5476U9X,2019-11-22,Spoke replacement (per spoke),11.49,MBCL24BW42
3,SRYHOJSQH6,CLJ8XEAY0C,2023-10-08,Chain replacement,27.78,MBCL24BW42
4,SRDQFQFYYR,CB46L6E7V3,2024-04-12,Pedal replacement,13.95,MA4MHCDLIX
...,...,...,...,...,...,...
7495,SRD4LZW94J,CHCZS03VUH,2021-08-28,"Tire replacement (per tire, excl. tire)",18.26,MV26A9TGJL
7496,SRU0A4MRD1,C6843EBZFN,2019-09-18,General inspection & safety check,45.40,MTA6ED59YC
7497,SR15JHZ4IV,CE6621OVX3,2021-02-07,"Tire replacement (per tire, excl. tire)",14.72,MQF1KPMUPR
7498,SRF8TMWJHG,CGGALTQYOE,2023-11-18,Gear shifting adjustment,26.25,M06GWF4R58


In [11]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_subvier = pd.read_sql_query('''
SELECT
ROUND(CAST(AVG(cost) AS NUMERIC), 2) AS AverageServiceCosts,
CAST(EXTRACT(YEAR FROM ServiceDate) AS INTEGER) AS Year
FROM servicerecords
WHERE servicedescription = 'Gear shifting adjustment'
GROUP BY Year
ORDER BY AverageServiceCosts DESC
''', con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_subvier

,averageservicecosts,year
0,25.83,2019
1,25.71,2021
2,25.40,2020
3,25.30,2022
4,25.07,2023
5,24.59,2024


#### <b>Aufgabe (5): Ermitteln Sie, wie viele Bicycles jeder Verkäufer verkauft hat und zu welchem Gesamtpreis.</b>
<b>Details zur Aufgabenstellung:</b>
- Benötigte Tabellen: Salespersons, Sales
- Benötigte Spalten:
  - aus Salespersons: SalespersonID, FirstName, LastName, Phone, Email
  - aus Sales: SaleID, SalePrice, SalespersonID
- Die Ergebnistabelle soll die Spalten FirstName, LastName, Phone, Email, TotalBicyclesSold, TotalSalesAmount enthalten.
- Sortieren Sie die Ausgabe absteigend nach TotalSalesAmount.

<b>Tipp:</b>
- In PostgreSQL ist ein JOIN standardmässig ein INNER JOIN.

<b style="color: gray;">(max. erreichbare Punkte: 8)</b>

In [12]:
df_salespersons

,salespersonid,firstname,lastname,phone,email,hiredate,commissionrate
0,SP86XCRI7B,Michael,Zhang,+41 (0)63 311 51 92,michael.zhang@example.com,2018-10-12,0.15
1,SPCZZUOCF9,Heather,Brown,+41 (0)46 288 75 19,heather.brown@example.com,2019-11-10,0.15
2,SPLO7OLQSK,Ryan,Baxter,+41 (0)40 706 53 70,ryan.baxter@example.com,2022-02-27,0.15
3,SPL5PTPR1D,Nicole,Carter,+41 (0)77 266 76 74,nicole.carter@example.com,2024-05-15,0.15
4,SPXRVM1B8C,Brian,Klein,+41 (0)77 449 26 62,brian.klein@example.com,2016-08-15,0.15
5,SPGK07EO84,Christine,Harris,+41 (0)61 523 86 58,christine.harris@example.com,2022-05-15,0.15
6,SPOHEVXP0T,Karen,Randall,+41 (0)66 616 17 40,karen.randall@example.com,2015-02-20,0.15
7,SP0LU6X4G6,Robert,Lyons,+41 (0)40 422 23 33,robert.lyons@example.com,2024-08-13,0.15
8,SPCGP1HZPG,Lauren,Robinson,+41 (0)71 214 12 13,lauren.robinson@example.com,2016-01-05,0.15
9,SPOELVJOXX,Cathy,Becker,+41 (0)23 263 98 44,cathy.becker@example.com,2021-08-08,0.15


In [13]:
df_sales

,saleid,bicycleid,customerid,saledate,saleprice,salespersonid
0,SLBF6MT8J2,C875HZ1AG6,CUIVBGDQ8B,2023-06-11,3480,SP0LU6X4G6
1,SQX72758TF,C2WFIK6WEP,CUZ6STRW2G,2019-09-15,2680,SPCZZUOCF9
2,S5FDZ50KNW,CGPCXD8911,CUGO27G9KZ,2021-03-05,3000,SP86XCRI7B
3,S79LBQUU41,C3U71M5BP5,CUNY08VOOD,2022-09-27,4320,SPCGP1HZPG
4,SIECZIGLOC,CLD9BSTE3H,CUJ52ZOJYZ,2023-02-14,1900,SPOHEVXP0T
...,...,...,...,...,...,...
679,STJ906IYGY,CALG7KLJHE,CUA32ROGBY,2020-08-01,3120,SP86XCRI7B
680,S1L1CI9G7K,C78URTU7TV,CUU7W491MR,2023-07-08,1550,SPCZZUOCF9
681,SJQ2VP6CY4,C1V12YL0CG,CUDJO2VSA8,2019-09-30,3360,SPOHEVXP0T
682,SJ3XI5MJ1Q,C6OURRJ88X,CU8SDBWV5E,2021-07-21,1630,SPXRVM1B8C


In [14]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_subfünf = pd.read_sql_query('''
SELECT 
salespersons.firstname,
salespersons.lastname,
salespersons.phone,
salespersons.email,
COUNT (*) AS TotalBicyclesSold,
SUM (sales.saleprice) AS TotalSalesAmount
FROM sales

INNER JOIN salespersons ON sales.salespersonid = salespersons.salespersonid

GROUP BY
salespersons.firstname,
salespersons.lastname,
salespersons.phone,
salespersons.email

ORDER BY TotalSalesAmount DESC
''', con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_subfünf

,firstname,lastname,phone,email,totalbicyclessold,totalsalesamount
0,Heather,Brown,+41 (0)46 288 75 19,heather.brown@example.com,82,267840.0
1,Christine,Harris,+41 (0)61 523 86 58,christine.harris@example.com,74,237420.0
2,Karen,Randall,+41 (0)66 616 17 40,karen.randall@example.com,75,232950.0
3,Robert,Lyons,+41 (0)40 422 23 33,robert.lyons@example.com,69,232330.0
4,Lauren,Robinson,+41 (0)71 214 12 13,lauren.robinson@example.com,69,216840.0
5,Ryan,Baxter,+41 (0)40 706 53 70,ryan.baxter@example.com,72,208490.0
6,Nicole,Carter,+41 (0)77 266 76 74,nicole.carter@example.com,66,207940.0
7,Cathy,Becker,+41 (0)23 263 98 44,cathy.becker@example.com,67,200340.0
8,Michael,Zhang,+41 (0)63 311 51 92,michael.zhang@example.com,59,180090.0
9,Brian,Klein,+41 (0)77 449 26 62,brian.klein@example.com,51,174610.0


#### <b>Aufgabe (6): Ermitteln Sie, welche Bicycles der Marke Bianchi von März bis Dezember 2023 verkauft wurden, zu welchem Verkaufspreis, an welche Kunden und durch welche Verkäufer.</b>
<b>Details zur Aufgabenstellung:</b>
- Benötigte Tabellen: Bicycles, Sales, Customers, Salespersons
- Benötigte Spalten:
  - aus Bicycles: BicycleID, Make, Model
  - aus Sales: BicycleID, CustomerID, SalespersonID, SalePrice, SaleDate
  - aus Customers: CustomerID, FirstName, LastName
  - aus Salespersons: SalespersonID, FirstName, LastName
- Die Ergebnistabelle soll die Spalten BicycleID, Make, Model, SalePrice, SaleDate, CustomerFirstName, CustomerLastName, SalespersonFirstName, SalespersonLastName enthalten.
- Sortieren Sie die Ausgabe aufsteigend nach SaleDate.
- Beschränken Sie die Ausgabe auf 10 Zeilen.

<b style="color: gray;">(max. erreichbare Punkte: 8)</b>

In [15]:
df_bicycles

,bicycleid,make,model,year,framenr,price,status
0,C7ZGWFLBUV,Trek,Domane,2019,A15 0LX,5066.94,In Maintenance
1,C875HZ1AG6,Merida,Reacto,2021,435N,5529.38,Sold
2,CDONEWSWFP,Merida,Big Nine,2021,QBA-572,1518.33,In Maintenance
3,C2UQZP803C,Trek,Marlin,2023,5-92778W,5038.99,In Maintenance
4,C2WFIK6WEP,Merida,Scultura,2020,300 VLT,1548.19,Sold
...,...,...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2020,911 LYI,2376.02,Available
1996,CP3F48BE9Y,Merida,Scultura,2022,YPU 135,2833.36,Available
1997,CHBXQ3SSNU,Merida,Scultura,2019,059-GNM,1629.91,In Maintenance
1998,CP5CTME2ZD,Trek,Marlin,2021,02V 6510,6176.55,In Maintenance


In [16]:
df_sales

,saleid,bicycleid,customerid,saledate,saleprice,salespersonid
0,SLBF6MT8J2,C875HZ1AG6,CUIVBGDQ8B,2023-06-11,3480,SP0LU6X4G6
1,SQX72758TF,C2WFIK6WEP,CUZ6STRW2G,2019-09-15,2680,SPCZZUOCF9
2,S5FDZ50KNW,CGPCXD8911,CUGO27G9KZ,2021-03-05,3000,SP86XCRI7B
3,S79LBQUU41,C3U71M5BP5,CUNY08VOOD,2022-09-27,4320,SPCGP1HZPG
4,SIECZIGLOC,CLD9BSTE3H,CUJ52ZOJYZ,2023-02-14,1900,SPOHEVXP0T
...,...,...,...,...,...,...
679,STJ906IYGY,CALG7KLJHE,CUA32ROGBY,2020-08-01,3120,SP86XCRI7B
680,S1L1CI9G7K,C78URTU7TV,CUU7W491MR,2023-07-08,1550,SPCZZUOCF9
681,SJQ2VP6CY4,C1V12YL0CG,CUDJO2VSA8,2019-09-30,3360,SPOHEVXP0T
682,SJ3XI5MJ1Q,C6OURRJ88X,CU8SDBWV5E,2021-07-21,1630,SPXRVM1B8C


In [17]:
df_customers

,customerid,firstname,lastname,phone,email,address
0,CUZG5MKZX0,Abigail,Hancock,+41 (0)62 599 21 88,abigail.hancock@example.com,"40694 Jason Route Suite 422, South Kevinfurt, ..."
1,CU6SCU6F8N,Ashley,Erickson,+41 (0)27 167 66 58,ashley.erickson@example.com,"179 Danielle Forge Apt. 682, Richardchester, M..."
2,CUPK9KDZPX,Ashley,Johnson,+41 (0)73 724 10 49,ashley.johnson@example.com,"220 Zachary Union, Leemouth, MT 05993"
3,CUBF9W5TPS,Mary,Wilson,+41 (0)59 200 15 78,mary.wilson@example.com,"2494 Lopez Canyon Suite 201, Leslieport, NH 38017"
4,CUSTG75SDB,Logan,Fritz,+41 (0)59 799 31 70,logan.fritz@example.com,"556 Eddie Parkways Apt. 754, Theresaview, IL 2..."
...,...,...,...,...,...,...
1995,CU8AB7W30L,Sharon,Rivas,+41 (0)70 817 47 73,sharon.rivas@example.com,"USNV Allen, FPO AE 89287"
1996,CUB60A9Z3H,Jeremy,Douglas,+41 (0)48 318 10 46,jeremy.douglas@example.com,"7664 Marshall Corner, North Lindsey, NM 16635"
1997,CUQTA7IQ93,Ann,Erickson,+41 (0)68 358 87 96,ann.erickson@example.com,"1056 Anderson Lights, West Peter, IN 05929"
1998,CUEM5CE0U6,Charles,White,+41 (0)28 953 92 13,charles.white@example.com,"69101 Moyer Plains Suite 430, Mccannchester, T..."


In [18]:
df_salespersons

,salespersonid,firstname,lastname,phone,email,hiredate,commissionrate
0,SP86XCRI7B,Michael,Zhang,+41 (0)63 311 51 92,michael.zhang@example.com,2018-10-12,0.15
1,SPCZZUOCF9,Heather,Brown,+41 (0)46 288 75 19,heather.brown@example.com,2019-11-10,0.15
2,SPLO7OLQSK,Ryan,Baxter,+41 (0)40 706 53 70,ryan.baxter@example.com,2022-02-27,0.15
3,SPL5PTPR1D,Nicole,Carter,+41 (0)77 266 76 74,nicole.carter@example.com,2024-05-15,0.15
4,SPXRVM1B8C,Brian,Klein,+41 (0)77 449 26 62,brian.klein@example.com,2016-08-15,0.15
5,SPGK07EO84,Christine,Harris,+41 (0)61 523 86 58,christine.harris@example.com,2022-05-15,0.15
6,SPOHEVXP0T,Karen,Randall,+41 (0)66 616 17 40,karen.randall@example.com,2015-02-20,0.15
7,SP0LU6X4G6,Robert,Lyons,+41 (0)40 422 23 33,robert.lyons@example.com,2024-08-13,0.15
8,SPCGP1HZPG,Lauren,Robinson,+41 (0)71 214 12 13,lauren.robinson@example.com,2016-01-05,0.15
9,SPOELVJOXX,Cathy,Becker,+41 (0)23 263 98 44,cathy.becker@example.com,2021-08-08,0.15


In [19]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_subsechs = pd.read_sql_query('''
SELECT
bicycles.bicycleid,
bicycles.make,
bicycles.model,
sales.saleprice,
sales.saledate,
customers.firstname AS CustomerFirstName,
customers.lastname AS CustomerLastName,
salespersons.firstname AS SalespersonFirstName,
salespersons.lastname AS SalespersonLastName
FROM bicycles

INNER JOIN sales
    ON bicycles.bicycleid = sales.bicycleid
INNER JOIN customers
    ON sales.customerid = customers.customerid
INNER JOIN salespersons
    ON sales.salespersonid = salespersons.salespersonid

WHERE make = 'Bianchi' AND saledate BETWEEN '2023-03-01' AND '2023-12-31'
    

ORDER BY SaleDate ASC
LIMIT 10
''', con = engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_subsechs

,bicycleid,make,model,saleprice,saledate,customerfirstname,customerlastname,salespersonfirstname,salespersonlastname
0,C28CVDTTSB,Bianchi,Oltre,1680,2023-04-27,Rebecca,Henderson,Christine,Harris
1,CINFLGUWPI,Bianchi,Aria,2220,2023-05-24,Jacqueline,Waters,Cathy,Becker
2,C0VVD5LONX,Bianchi,Oltre,2360,2023-06-06,Jesus,Johnston,Christine,Harris
3,C9FCLAZ5BS,Bianchi,Aria,3020,2023-06-16,Emma,Stewart,Michael,Zhang
4,CVSWF385E4,Bianchi,Oltre,2460,2023-07-31,Jonathan,Camacho,Nicole,Carter
5,CJ21FVDXKP,Bianchi,Infinito,1700,2023-08-31,Amy,Gomez,Michael,Zhang
6,C4Y0NPNTOK,Bianchi,Aria,4660,2023-09-16,Anna,Phillips,Heather,Brown
7,CXYG3TWNY0,Bianchi,Aria,2070,2023-09-30,Denise,Jones,Christine,Harris
8,CZ2M1IWTJO,Bianchi,Oltre,4860,2023-10-14,Christina,Morgan,Brian,Klein
9,C1U99AVROS,Bianchi,Oltre,3120,2023-10-24,Patrick,Lopez,Robert,Lyons


#### <b>Aufgabe (7): Ermitteln Sie über alle Jahre hinweg (nicht pro Jahr), wie hoch die durchschnittlichen Servicekosten pro Mechaniker und Fahrradmarke waren.</b>
<b>Details zur Aufgabenstellung:</b>
- Benötigte Tabellen: Mechanics, ServiceRecords, Bicycles
- Benötigte Spalten:
  - aus Mechanics: MechanicID, FirstName, LastName
  - aus ServiceRecords: MechanicID, BicycleID, Cost, ServiceDate
  - aus Bicycles: BicycleID, Make
- Die Ergebnistabelle soll die Spalten MechanicFirstName, MechanicLastName, Make, ServicesCount, AverageServiceCosts, LastServiceDate enthalten.
- Runden Sie AverageServiceCosts auf zwei Nachkommastellen.
- Sortieren Sie die Ausgabe absteigend nach AverageServiceCosts.

<b>Tipp:</b>
- LastServiceDate kann mit Hilfe von MAX(sr.ServiceDate) AS LastServiceDate ermittelt werden (sr = Tabelle ServiceRecords).

<b style="color: gray;">(max. erreichbare Punkte: 8)</b>

In [20]:
df_mechanics

,mechanicid,firstname,lastname,phone,email,hiredate,specialization
0,MV26A9TGJL,Erin,Williamson,+41 (0)65 305 62 20,erin.williamson@example.com,2019-01-31,Brakes
1,MTA6ED59YC,Melissa,Miller,+41 (0)77 431 12 29,melissa.miller@example.com,2022-11-12,Tires
2,MBCL24BW42,Bryan,White,+41 (0)23 791 68 46,bryan.white@example.com,2017-12-29,Elektrik
3,M8DA1H43YN,Shannon,Larsen,+41 (0)41 390 74 67,shannon.larsen@example.com,2016-08-18,Brakes
4,M8XUQMDR0P,Courtney,Ford,+41 (0)50 651 77 58,courtney.ford@example.com,2016-07-27,General
5,M06GWF4R58,Tiffany,Wilson,+41 (0)22 475 52 27,tiffany.wilson@example.com,2017-07-11,Crankset
6,MLNW721B7L,Bradley,Mckinney,+41 (0)23 859 32 52,bradley.mckinney@example.com,2023-02-07,Elektrik
7,MA4MHCDLIX,Cassandra,Sanford,+41 (0)26 322 84 52,cassandra.sanford@example.com,2016-12-23,Gear
8,MCNI3BOSV1,Lisa,Hall,+41 (0)56 904 84 24,lisa.hall@example.com,2023-02-18,General
9,MQF1KPMUPR,Cynthia,Pittman,+41 (0)39 208 58 20,cynthia.pittman@example.com,2023-12-02,Elektrik


In [21]:
df_servicerecords

,serviceid,bicycleid,servicedate,servicedescription,cost,mechanicid
0,SRPK835UWM,CRK1AC8K7M,2020-02-21,General inspection & safety check,58.60,MTA6ED59YC
1,SR4CZM07VB,C88VOY5YSJ,2019-10-27,"Tire replacement (per tire, excl. tire)",17.24,MCNI3BOSV1
2,SRKMA0NOB6,CVS5476U9X,2019-11-22,Spoke replacement (per spoke),11.49,MBCL24BW42
3,SRYHOJSQH6,CLJ8XEAY0C,2023-10-08,Chain replacement,27.78,MBCL24BW42
4,SRDQFQFYYR,CB46L6E7V3,2024-04-12,Pedal replacement,13.95,MA4MHCDLIX
...,...,...,...,...,...,...
7495,SRD4LZW94J,CHCZS03VUH,2021-08-28,"Tire replacement (per tire, excl. tire)",18.26,MV26A9TGJL
7496,SRU0A4MRD1,C6843EBZFN,2019-09-18,General inspection & safety check,45.40,MTA6ED59YC
7497,SR15JHZ4IV,CE6621OVX3,2021-02-07,"Tire replacement (per tire, excl. tire)",14.72,MQF1KPMUPR
7498,SRF8TMWJHG,CGGALTQYOE,2023-11-18,Gear shifting adjustment,26.25,M06GWF4R58


In [22]:
df_bicycles

,bicycleid,make,model,year,framenr,price,status
0,C7ZGWFLBUV,Trek,Domane,2019,A15 0LX,5066.94,In Maintenance
1,C875HZ1AG6,Merida,Reacto,2021,435N,5529.38,Sold
2,CDONEWSWFP,Merida,Big Nine,2021,QBA-572,1518.33,In Maintenance
3,C2UQZP803C,Trek,Marlin,2023,5-92778W,5038.99,In Maintenance
4,C2WFIK6WEP,Merida,Scultura,2020,300 VLT,1548.19,Sold
...,...,...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2020,911 LYI,2376.02,Available
1996,CP3F48BE9Y,Merida,Scultura,2022,YPU 135,2833.36,Available
1997,CHBXQ3SSNU,Merida,Scultura,2019,059-GNM,1629.91,In Maintenance
1998,CP5CTME2ZD,Trek,Marlin,2021,02V 6510,6176.55,In Maintenance


In [23]:

# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# SQL-Abfrage erstellen und ausführen
df_subsieben = pd.read_sql_query('''
SELECT
mechanics.firstname AS MechanicFirstName,
mechanics.lastname AS MechanicLastName,
bicycles.Make,
COUNT (*) AS ServicesCount,
ROUND(CAST(AVG(cost) AS NUMERIC), 2) AS AverageServiceCosts,
MAX(servicerecords.ServiceDate) AS LastServiceDate
FROM mechanics

INNER JOIN servicerecords
    ON mechanics.mechanicid = servicerecords.mechanicid
INNER JOIN bicycles
    ON servicerecords.bicycleid = bicycles.bicycleid

GROUP BY
mechanics.firstname,
mechanics.lastname,
bicycles.make

ORDER BY AverageServiceCosts DESC
''', con=engine)

# Datenbankverbindung trennen
engine.dispose()

# Ergebnis anzeigen
df_subsieben

,mechanicfirstname,mechaniclastname,make,servicescount,averageservicecosts,lastservicedate
0,Shannon,Larsen,Scott,65,39.76,2024-08-07
1,Cynthia,Pittman,Cannondale,74,38.61,2024-07-28
2,Shannon,Larsen,Specialized,75,37.41,2024-08-20
3,Courtney,Ford,Santa Cruz,82,36.41,2024-08-04
4,Erin,Williamson,Specialized,69,36.14,2024-08-17
...,...,...,...,...,...,...
95,Cynthia,Pittman,Specialized,82,27.53,2024-08-19
96,Lisa,Hall,Bianchi,70,27.47,2024-08-26
97,Cynthia,Pittman,Bianchi,77,26.81,2024-07-20
98,Melissa,Miller,Bianchi,71,26.49,2024-08-26


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [24]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')


-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-10-14 23:22:02
Python Version: 3.12.11
IP Address: 127.0.0.1
-----------------------------------
